In [ ]:
SPEC={'model': 'Qwen/Qwen3-Embedding-8B', 'source_kernel': 'ntklinhfitus/medxir-qwen8b-additions-20261030-r2', 'embedding_manifest_sha256': '4e3e8d6c3648b3f31395b9ae4966d15a7fc2afe9ef9f7847077e11825b08035b', 'corpus_manifest_sha256': 'fdc057d88448532cd9565209135c65334f640f27ab6b45324cd76368b00bb1a7', 'counts': {'vi': 95938, 'en': 14, 'zh': 174453}}
SOURCE='"""Kaggle CPU: reorder existing vectors into the three corpus-v2 language parts."""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport hashlib,json,shutil,zipfile\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nKEYS={\'vi\':\'vi-part-007\',\'en\':\'en-part-001\',\'zh\':\'zh-part-001\'}\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as h:return hashlib.file_digest(h,\'sha256\').hexdigest()\n\n\ndef stats(vectors):\n    lo,hi=2.,0.\n    for start in range(0,len(vectors),8192):\n        block=np.array(vectors[start:start+8192],dtype=np.float32)\n        assert np.isfinite(block).all()\n        norms=np.linalg.norm(block,axis=1)\n        assert np.max(abs(norms-1))<0.01\n        lo=min(lo,float(norms.min()));hi=max(hi,float(norms.max()))\n    return dict(nan_rows=0,norm_min=lo,norm_max=hi)\n\n\ndef package(source,corpus,out,spec):\n    source,corpus,out=map(Path,(source,corpus,out));(out/\'vectors\').mkdir(parents=True,exist_ok=True)\n    assert sha(source/\'incremental_embed_manifest.json\')==spec[\'embedding_manifest_sha256\']\n    assert sha(corpus/\'chunk_manifest.json\')==spec[\'corpus_manifest_sha256\']\n    parent=json.loads((source/\'incremental_embed_manifest.json\').read_text())\n    assert parent[\'complete\'] and parent[\'model\']==spec[\'model\'] and parent[\'dim\']==4096\n    assert parent[\'max_length\']==1024 and parent[\'output_dtype\']==\'float16\'\n    targets={};arrays={};seen={};lookup={}\n    for lang,key in KEYS.items():\n        path=corpus/f\'chunks/{lang}/{key.removeprefix(lang+"-")}.parquet\'\n        records=pq.read_table(path,columns=[\'chunk_id\',\'lang\']).to_pydict()\n        ids=records[\'chunk_id\'];assert set(records[\'lang\'])=={lang} and len(ids)==spec[\'counts\'][lang]\n        targets[lang]=ids;seen[lang]=np.zeros(len(ids),dtype=bool)\n        arrays[lang]=np.lib.format.open_memmap(out/f\'vectors/{key}.npy\',mode=\'w+\',dtype=np.float16,shape=(len(ids),4096))\n        for pos,cid in enumerate(ids):\n            assert cid not in lookup;lookup[cid]=(lang,pos)\n    assert len(lookup)==parent[\'total_rows\']==sum(spec[\'counts\'].values())\n    copied=0\n    for part in parent[\'parts\']:\n        for kind in [\'vectors\',\'ids\']:\n            meta=part[kind];path=source/meta[\'path\']\n            assert path.stat().st_size==meta[\'bytes\'] and sha(path)==meta[\'sha256\']\n        vectors=np.load(source/part[\'vectors\'][\'path\'],mmap_mode=\'r\',allow_pickle=False)\n        records=pq.read_table(source/part[\'ids\'][\'path\'],columns=[\'chunk_id\',\'lang\']).to_pylist()\n        assert vectors.dtype==np.float16 and vectors.shape==(len(records),4096)\n        mappings=defaultdict(lambda:([],[]))\n        for row,record in enumerate(records):\n            cid=record[\'chunk_id\'];assert cid in lookup\n            lang,pos=lookup[cid];assert lang==record[\'lang\'] and not seen[lang][pos]\n            seen[lang][pos]=True;mappings[lang][0].append(row);mappings[lang][1].append(pos)\n        for lang,(src_rows,dst_rows) in mappings.items():\n            block=np.array(vectors[src_rows],copy=True)\n            arrays[lang][dst_rows]=block\n            assert np.array_equal(arrays[lang][dst_rows].view(np.uint16),block.view(np.uint16)),\'Vector bytes changed during reorder\'\n        copied+=len(records);del vectors\n        print(f\'Reordered {copied:,}/{len(lookup):,} existing vector rows\',flush=True)\n    assert all(flags.all() for flags in seen.values())\n    parts=[]\n    for lang,key in KEYS.items():\n        arrays[lang].flush()\n        checks=stats(arrays[lang])\n        pq.write_table(pa.table({\'chunk_id\':targets[lang]}),out/f\'vectors/{key}-ids.parquet\',compression=\'zstd\')\n        saved=pq.read_table(out/f\'vectors/{key}-ids.parquet\')[\'chunk_id\'].to_pylist()\n        assert saved==targets[lang]\n        parts.append(dict(part=key,lang=lang,chunks=len(saved),**checks))\n        arrays[lang]._mmap.close()\n        print(f\'PASS {key}: {len(saved):,} aligned rows; values unchanged\',flush=True)\n    report=dict(model=parent[\'model\'],model_revision=parent[\'model_revision\'],dim=4096,dtype=parent[\'dtype\'],\n        output_dtype=\'float16\',doc_prefix=parent[\'doc_prefix\'],batch=parent[\'batch\'],max_length=1024,instruction=parent[\'instruction\'],\n        pooling=parent[\'pooling\'],padding_side=parent[\'padding_side\'],normalize=parent[\'normalize\'],\n        complete=True,parts_done=3,parts_total=3,chunks=copied,nan_rows=0,parts=parts,\n        new_model_forward_inputs=0,repacked_without_inference=True,vectors_bitwise_preserved=True,\n        all_ids_match_corpus_part_order=True,baseline_modified=False,\n        source_embedding_manifest_sha256=spec[\'embedding_manifest_sha256\'],corpus_manifest_sha256=spec[\'corpus_manifest_sha256\'],\n        corpus_dataset=\'ntklinhfitus/medxir-chunk-corpus-v2\',corpus_version=\'v2\',corpus_dataset_version=1,\n        source_plan_sha256=parent.get(\'source_plan_sha256\'),\n        model_package_manifest_sha256=parent.get(\'model_package_manifest_sha256\'),\n        source_inference_actions=parent.get(\'actions\'),\n        source_forward_inputs_including_compatibility=parent.get(\'forward_tally_including_compatibility\',{}).get(\'forward_inputs\'),\n        compatibility_cosine_min=parent.get(\'compatibility_cosine_min\'),\n        baseline_chunks_verified=parent.get(\'baseline_chunks_verified\'),\n        source_kernel=spec[\'source_kernel\'],\n        files={p.relative_to(out).as_posix():dict(bytes=p.stat().st_size,sha256=sha(p)) for p in sorted((out/\'vectors\').iterdir())})\n    (out/\'embed_manifest.json\').write_text(json.dumps(report,indent=2))\n    return report\n\n\ndef main():\n    spec=json.loads(Path(\'/kaggle/working/vector_repack_spec.json\').read_text())\n    sources=list(Path(\'/kaggle/input\').rglob(\'incremental_embed_manifest.json\'))\n    corpora=list(Path(\'/kaggle/input\').rglob(\'chunk_manifest.json\'))\n    assert len(sources)==len(corpora)==1\n    out=Path(\'/kaggle/working/vector_payload\')\n    report=package(sources[0].parent,corpora[0].parent,out,spec)\n    files={p.relative_to(out).as_posix():dict(bytes=p.stat().st_size,sha256=sha(p)) for p in out.rglob(\'*\') if p.is_file()}\n    validation=dict(complete=True,model=report[\'model\'],dim=4096,chunks=report[\'chunks\'],parts=3,new_model_forward_inputs=0,\n        vectors_bitwise_preserved=True,all_ids_match_corpus_part_order=True,files=files)\n    Path(\'/kaggle/working/vector_repack_validation.json\').write_text(json.dumps(validation,indent=2))\n    print(json.dumps({k:v for k,v in validation.items() if k!=\'files\'},indent=2))\n\n\n\nif __name__==\'__main__\':main()\n'
from pathlib import Path
import json, subprocess, sys
p=Path('/kaggle/working/repack_8b_additions.py'); p.write_text(SOURCE, encoding='utf-8')
Path('/kaggle/working/vector_repack_spec.json').write_text(json.dumps(SPEC))
subprocess.run([sys.executable, '-u', str(p)], check=True)
